In [1512]:
"""
This script creates artificial data for a discrete choice problem.
Assume there are three modes of transportation to choose from. Six fixed
variables were designed as significant and five as non-significant.
Seven random variables (five normal, one uniform, one triangular) were
designed as significant.
Three normal variables were correlated.
Two normal variables were non-linearly transformed.
"""


'\nThis script creates artificial data for a discrete choice problem.\nAssume there are three modes of transportation to choose from. Six fixed\nvariables were designed as significant and five as non-significant.\nSeven random variables (five normal, one uniform, one triangular) were\ndesigned as significant.\nThree normal variables were correlated.\nTwo normal variables were non-linearly transformed.\n'

In [1513]:

import numpy as np

import random





import statsmodels.api as sm
from scipy.stats import multivariate_normal

In [1514]:
import numpy as np
import pandas as pd
import scipy.stats as ss

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
is_correlated = 0

    
if is_correlated:
    name_2 = 'artificial_mixed_corr_2023_MOOF.csv'
else:
    name_2 = 'artificial_ZA.csv'
    
is_panel = 1
if is_panel:
    name_2 = 'panel_synth.csv'


In [1515]:
def noise(n_obs, perc=1, random_state=None):
    random_state = random_state or np.random
    noise_vec = random_state.normal(0, 1, n_obs)
    return noise_vec

In [1516]:
def random_col(N, P, J, random_state=None, low = 5, high = 25, noise_val  = 0):
    rand_nums = random_state.randint(low=low, high=high, size=(N,))/4
    return np.repeat(rand_nums, P) + noise_val*noise(N*P*J, random_state=random_state)

def generate_random_df(N, P, J, num_fixed=0, num_isvars=0, num_randvars=0, random_state=None, G = None):
    df = pd.DataFrame()
    #df['choice_id'] = np.repeat(np.arange(1, (N*P+1)), J)
    df['ind_id'] = np.repeat(np.arange(1, N+1), P*J)
    #df['alt'] = np.tile(np.arange(1, J+1), N*P)

    varnames = []
    
    
    for i in range(num_isvars):
        coef_name = 'added_isvar' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, low = 0, high = 10, noise_val = .5)
    
    
    for i in range(num_fixed):
        coef_name = 'added_fixed' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, low = 0, high = 5)

    
        

    for i in range(num_randvars):
        coef_name = 'added_random' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, low = 0, high =5)
        
    for i in range(1):
        coef_name = 'constant'
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, low = 1, high =2)*4    
    if G is not None:
        df['group'] = np.repeat(np.arange(1, N/G+1),P*J*G)

    return df, varnames


In [1517]:

N =1000  # Number of observations
P = 1  # Number of choices per individual
J = 1  # Number of alternatives
G =4
num_fixed = 0
num_isvars = 3
num_nonsig = 3
num_randvars = 3
seed = 122
random_state = np.random.RandomState(seed)
np.random.seed(seed)

#random.seed(seed)
df, varnames = generate_random_df(N, P, J, num_fixed=num_fixed, num_isvars=num_isvars,
                                  num_randvars=num_randvars, random_state=random_state, G = 5)

# Shift values <-2 for as inverse boxcox transform will be applied



np.shape(df)

(1000, 9)

In [1518]:
df.head(40)

,ind_id,added_isvar1,added_isvar2,added_isvar3,added_random1,added_random2,added_random3,constant,group
0,1,0.177677,0.863979,-0.355984,1.00,0.00,0.00,1.0,1.0
1,2,-0.489710,2.069492,0.900920,0.25,1.00,0.00,1.0,1.0
2,3,-0.311230,0.440724,0.722574,0.25,1.00,0.50,1.0,1.0
3,4,1.355794,1.634756,2.269609,0.75,0.50,1.00,1.0,1.0
4,5,1.380570,0.560777,0.735976,0.75,0.75,1.00,1.0,1.0
5,6,1.657335,0.871882,0.319339,0.25,0.00,0.75,1.0,2.0
6,7,1.488775,2.637729,-0.067471,1.00,0.50,0.75,1.0,2.0
7,8,0.761594,1.094125,-0.016147,0.50,0.75,0.50,1.0,2.0
8,9,1.408956,1.943611,1.539711,0.25,1.00,1.00,1.0,2.0
9,10,1.497488,0.615846,1.573195,0.25,1.00,0.25,1.0,2.0


In [1519]:
# Define coefficients (betas)
# Fixed betas
fixed_coefs = [ random_state.choice([1,2,3 , 3, 2,3]) *random_state.uniform(1, 4) for i in range(num_fixed)]
fixed_coefs = np.array(fixed_coefs)



fixed_coefs = list(fixed_coefs)


In [1520]:
# Random mean between -1.5 and 1.5, excluding -.1 - .1 as hard to detect effect
random_coefs_mean = [random_state.choice([1,6, 5, 7,4, 2, 2, 3, 4]) * random_state.uniform(.5, 2.5) for i in range(num_randvars)]
random_coefs_sd = [random_state.uniform(2.0, 4) for i in range(num_randvars)]
print(random_coefs_sd)
cov_mat = np.diag(random_coefs_sd)
cov_mat[0, 1] = cov_mat[1, 0] = 2.25
cov_mat[0, 2] = cov_mat[2, 0] = 0.4
cov_mat[1, 2] = cov_mat[2, 1] = 1.8

random_coefs_uniform_a = 0
random_coefs_uniform_b = random_state.uniform(2, 4)

random_coefs_tri_left = 0
random_coefs_tri_right = random_state.uniform(1, 2)
random_coefs_tri_mode = random_coefs_tri_right/2

rand_coefs = [np.array([]) for i in range(num_randvars)]

a = int(N/G)
draws = list()
ndraws = 50
for rr in range(ndraws):
    
    for i in range(a):
        res_normal = random_state.multivariate_normal(random_coefs_mean, cov_mat)
        res_uniform = np.array([random_state.uniform(random_coefs_uniform_a, random_coefs_uniform_b)])
        res_triangular = np.array([random_state.triangular(random_coefs_tri_left, random_coefs_tri_mode, random_coefs_tri_right)])
        res = np.concatenate((res_normal, res_uniform, res_triangular))

        for r in range(num_randvars):
        # print(res[r])
            
            rand_coefs[r] = np.append(rand_coefs[r], np.repeat(res[r], P*J*G))
            #print(rand_coefs[r])
    draws.append(rand_coefs) 
    rand_coefs = [np.array([]) for i in range(num_randvars)]     


[2.4081052655738704, 3.295878115620999, 2.1954073312625955]


In [1521]:
print(np.shape(draws))
draws = np.array(draws).T

print(np.shape(draws))
rand_coefs =draws

(50, 3, 1000)
(1000, 3, 50)


In [1522]:
random_coefs_uniform_a, random_coefs_uniform_b

(0, 2.2667731150830828)

In [1523]:
random_coefs_tri_left, random_coefs_tri_mode, random_coefs_tri_right

(0, 0.5492896945756325, 1.098579389151265)

In [1524]:
np.size(rand_coefs)
P*N*J
np.shape(rand_coefs)

(1000, 3, 50)

In [1525]:
B_fixed = [np.repeat(f_coef, P*N*J) for f_coef in fixed_coefs]
B_const = [np.repeat(3, P*N*J)]
if is_correlated:
    B_const = [np.repeat(-3, P*N*J)]
else:
    B_const = [np.repeat(-47, P*N*J)]
    


# Convert betas to matrix for easy product


In [1526]:
# Visualise values after non-linear transformation
# import matplotlib.pyplot as plt

B_const = np.array(B_const).T
rand_co

(1000, 1)


In [1527]:
# Multiply and generate probability
#isvars = ['added_isvar' + str(i+1) for i in range(num_isvars)]
Xf = df.values[:, num_fixed+num_randvars+num_isvars+1:num_fixed+num_randvars+2+num_isvars] 
Xr = df.values[:, 1+num_isvars:num_fixed+num_randvars+2+num_isvars-1]  # Extract only necessary columns
print(Xf.shape)
rand_coefs = np.array(rand_coefs)
print(np.shape(rand_coefs))
#XB =      (X*B).sum(axis = 1).reshape(N*P, J)
#result = np.einsum('ij,ij->ij', a, b)

print(B_const.shape, 'plea')
#Vdf = np.einsum('nk,nk -> n,k', Xf, B_const) # (N, K)
Vdr = np.einsum("nk,nkr -> nr", Xr, rand_coefs)  # (N,P,R)
print(Vdr.shape, 
      'd')
Vdboth = B_const[:, :, None] + Vdr[:, None, :]
print(Vdboth.shape, 'f')
XB = np.exp(np.clip(Vdboth, None, 700)).sum(axis = 2)/ndraws
scale = 1
dispersion = 3

eps = np.random.gumbel(0, 1, (N*P, J))
print(max(XB))
#XB = np.clip(XB, None, 7)
eXB = np.exp(XB).ravel()
dispersion = 10
scale = 1



#xg = np.array(rgamma(N, dispersion, dispersion))
xbg = eXB
#xbg = eXB

print(max(eXB))
nbyo = np.random.poisson(xbg)

# Use monte carlo simulation to predict choice
# y = np.apply_along_axis(lambda p: np.eye(J, dtype='int64')[np.argmax(p)], 1, prob).reshape(N*P*J,)
# y = y.reshape(N*P*J,)
print('max is', max(nbyo))
y = []
U = XB + eps


df['Y'] = nbyo
count = len(list(filter(lambda x: x != 0, nbyo)))
print(count)  # Output: 4


# Apply non-linear transformations for boxcox testing
save = "C:/Users/n9471103/source/repos/MetaCount/" +name_2
# Save to CSV
print(df.head())
#df = df.drop(columns = ['id'])
print(df.head())
df.to_csv(save, index=False)

(1000, 1)
(1000, 3, 50)
(1000, 1) plea
(1000, 50) d
(1000, 1, 50) f
[1.55591844e-05]
1.0000155593054472
max is 6
666
   ind_id  added_isvar1  added_isvar2  added_isvar3  added_random1  \
0       1      0.177677      0.863979     -0.355984           1.00   
1       2     -0.489710      2.069492      0.900920           0.25   
2       3     -0.311230      0.440724      0.722574           0.25   
3       4      1.355794      1.634756      2.269609           0.75   
4       5      1.380570      0.560777      0.735976           0.75   

   added_random2  added_random3  constant  group  Y  
0           0.00            0.0       1.0    1.0  1  
1           1.00            0.0       1.0    1.0  0  
2           1.00            0.5       1.0    1.0  2  
3           0.50            1.0       1.0    1.0  0  
4           0.75            1.0       1.0    1.0  1  
   ind_id  added_isvar1  added_isvar2  added_isvar3  added_random1  \
0       1      0.177677      0.863979     -0.355984           1.00  

In [1528]:
# np.linalg.norm(model.coeff_[:11] - fixed_coefs)